<a href="https://colab.research.google.com/github/JJOM08/Proyectos-Jave/blob/main/SNAKE_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1. IMPORTAR LAS LIBRERÍAS
import random  # Para generar posiciones aleatorias (comida, obstáculos)
import os # Para limpiar la pantalla en consola
from collections import deque # Para manejar la serpiente como una cola doble

In [ ]:
import random # Importa random para que sea accesible en esta celda
import os # Importa os para que sea accesible en esta celda (para limpiar pantalla)
from collections import deque # Importa deque aquí para que sea accesible en esta celda

class JuegoSerpiente:
    def __init__(self, ancho=24, alto=17):
        self.ancho = ancho
        self.alto = alto
        self.serpiente = deque([(ancho//2, alto//2)])   # Tú controlas esta, 'deque' es eficiente para añadir/quitar de ambos extremos
        self.direccion = (1, 0)                         # Dirección inicial: hacia la derecha (cambio en x, cambio en y)
        self.comida = None                              # Posición de la comida
        self.obstaculos = set()                         # Conjunto de obstáculos para búsquedas rápidas
        self.puntaje = 0
        self.viva = True

        self.colocar_comida()                           # Genera la primera comida
        self.generar_obstaculos(10)                     # Genera obstáculos iniciales
        self.enemigo = SerpienteIA(ancho, alto)         # Inicializa la serpiente enemiga (IA)

    def generar_obstaculos(self, cantidad=10):
        # Genera una lista de todas las posiciones posibles para los obstáculos
        # Los obstáculos se generan dentro de un margen (3 unidades desde cada borde)
        # y no pueden superponerse con la serpiente al inicio del juego.
        posibles_obstaculos = [
            (x, y)
            for x in range(3, self.ancho - 3)
            for y in range(3, self.alto - 3)
            if (x, y) not in self.serpiente
        ]

        # Asegura que no se pidan más obstáculos de los que hay disponibles
        num_obstaculos_a_generar = min(cantidad, len(posibles_obstaculos))

        # Selecciona aleatoriamente la cantidad de obstáculos de las posiciones posibles
        self.obstaculos = set(random.sample(posibles_obstaculos, num_obstaculos_a_generar))

    def colocar_comida(self):
      while True:
        # Genera una posición aleatoria para la comida en todo el tablero
        pos = (random.randint(0, self.ancho-1), random.randint(0, self.alto-1))
        # Asegura que la comida no aparezca sobre la serpiente ni un obstáculo
        if pos not in self.serpiente and pos not in self.obstaculos:
            self.comida = pos
            break

    def dibujar(self):
        os.system('cls' if os.name == 'nt' else 'clear')  # Limpia pantalla de la consola (Windows o Linux/macOS)

        # Marco superior de la interfaz
        print("╔" + "═" * (self.ancho * 2 + 4) + "╗")
        print("║" + "   🐍 SNAKE ".center(self.ancho*2 + 4) + "║") # Título centrado
        print("╠" + "═" * (self.ancho * 2 + 4) + "╣")

        # Recorre cada fila y columna del tablero para dibujar los elementos
        for y in range(self.alto):
            fila = [
                ("🟢" if (x, y) == self.serpiente[0] else
                 "🔵" if (x, y) in self.serpiente else
                 "🐍" if self.enemigo and (x, y) == self.enemigo.serpiente[0] else
                 "🔴" if self.enemigo and (x, y) in self.enemigo.serpiente else
                 "⬛" if (x, y) in self.obstaculos else
                 "🍎" if (x, y) == self.comida else
                 "⬜")
                for x in range(self.ancho)
            ]
            print("║ " + " ".join(fila) + " ║") # Imprime la fila con un marco lateral

        # Marco inferior y estadísticas del juego
        print("╚" + "═" * (self.ancho * 2 + 4) + "╝")
        print(f"   **TÚ** → Puntaje: {self.puntaje}   |   Largo: {len(self.serpiente)}")
        print(f"   Enemigo → Largo: {len(self.enemigo.serpiente)}")
        print("\nControles:  W = ↑    S = ↓    A = ←    D = →")
        print("Escribe la letra y presiona ENTER (Q = Salir)")

    def paso(self, tecla):
        if not self.viva:
            return False # Si la serpiente no está viva, no puede moverse

        # Mapeo de teclas a direcciones (cambio en x, cambio en y)
        dirs = {'w': (0, -1), 's': (0, 1), 'a': (-1, 0), 'd': (1, 0)}

        if tecla in dirs:
            nueva_dir = dirs[tecla]
            # Evita giro de 180°: si la nueva dirección es opuesta a la actual, no la permite
            if (nueva_dir[0] != -self.direccion[0]) or (nueva_dir[1] != -self.direccion[1]):
                self.direccion = nueva_dir
        else:
            return False  # Tecla inválida, no se procesa el movimiento

        cabeza = self.serpiente[0] # Posición actual de la cabeza
        # Calcula la nueva posición de la cabeza basándose en la dirección
        nueva_cabeza = (cabeza[0] + self.direccion[0], cabeza[1] + self.direccion[1])

        # Detección de colisiones:
        # 1. Con los bordes del tablero
        # 2. Con su propio cuerpo (excepto el último segmento que se moverá)
        # 3. Con los obstáculos
        if (nueva_cabeza[0] < 0 or nueva_cabeza[0] >= self.ancho or
            nueva_cabeza[1] < 0 or nueva_cabeza[1] >= self.alto or
            nueva_cabeza in self.serpiente or nueva_cabeza in self.obstaculos):
            self.viva = False # La serpiente muere
            return False

        self.serpiente.appendleft(nueva_cabeza)  # Añade la nueva cabeza al frente de la serpiente

        if nueva_cabeza == self.comida:
            self.puntaje += 25                   # Suma puntos si come
            self.colocar_comida()                # Genera una nueva comida
        else:
            self.serpiente.pop()                 # Si no come, quita el último segmento (la serpiente se mueve sin crecer)

        return True

In [ ]:
#CLASE ENEMIGO (IA)
from collections import deque # Importa deque para que sea accesible en esta clase
class SerpienteIA:
    def __init__(self, ancho, alto):
        self.serpiente = deque([(5, 5)])         # Posición inicial de la IA
        self.ancho = ancho
        self.alto = alto

    def paso(self, comida):
        cabeza = self.serpiente[0] # Posición actual de la cabeza de la IA
        # Calcula la dirección hacia la comida:
        # dx: -1 si la comida está a la izquierda, 1 si está a la derecha, 0 si está en la misma columna
        # dy: -1 si la comida está arriba, 1 si está abajo, 0 si está en la misma fila
        dx = 1 if comida[0] > cabeza[0] else -1 if comida[0] < cabeza[0] else 0
        dy = 1 if comida[1] > cabeza[1] else -1 if comida[1] < cabeza[1] else 0

        # Prioriza el movimiento horizontal si es posible, si no, vertical
        dir_ia = (dx, 0) if dx != 0 else (0, dy)
        # Calcula la potencial nueva posición de la cabeza de la IA
        nueva = (cabeza[0] + dir_ia[0], cabeza[1] + dir_ia[1])

        # Si la dirección preferida es inválida (choca con borde o su propio cuerpo),
        # busca una dirección alternativa (arriba, abajo, derecha, izquierda)
        if not (0 <= nueva[0] < self.ancho and 0 <= nueva[1] < self.alto and nueva not in self.serpiente):
            # Busca la primera dirección alternativa válida usando una expresión generadora
            # y 'next()' para detenerse en la primera coincidencia.
            # Si no encuentra ninguna alternativa válida, mantiene 'dir_ia' como estaba.
            alternativas = (
                d for d in [(0,1),(0,-1),(1,0),(-1,0)] # Pruebas las 4 direcciones cardinales
                if (
                    0 <= (cabeza[0] + d[0]) < self.ancho and  # Verifica que la nueva posición en X esté dentro del tablero (no salga por la izquierda ni la derecha)
                    0 <= (cabeza[1] + d[1]) < self.alto and #Lo mismo con la nueva posición en y
                    (cabeza[0] + d[0], cabeza[1] + d[1]) not in self.serpiente
                )
            )

            dir_ia = next(alternativas, dir_ia) # Obtiene la primera alternativa o mantiene la actual
            nueva = (cabeza[0] + dir_ia[0], cabeza[1] + dir_ia[1]) # Recalcula 'nueva' con la dirección elegida

        self.serpiente.appendleft(nueva)         # Añade la nueva cabeza al frente de la serpiente IA
        if len(self.serpiente) > 1:
            self.serpiente.pop()                 # Mantiene el tamaño de la serpiente IA fijo (no crece al 'comer')

In [ ]:

#INICIO DEL JUEGO
juego = JuegoSerpiente() # Crea una instancia del juego

print("🎮 ¡Bienvenido! Tú eres la serpiente con cabeza **🟢** verde\n")

while juego.viva: # El bucle principal del juego continúa mientras la serpiente del jugador esté viva
    juego.dibujar() # Dibuja el estado actual del tablero
    tecla = input("\n→ Tu movimiento (w/a/s/d) o Q para salir: ").strip().lower() # Solicita la entrada del jugador

    if tecla == 'q':
        print("¡Gracias por jugar!")
        break # Sale del bucle si el jugador presiona 'q'

    if tecla not in ['w', 'a', 's', 'd']:
        print("❌ Letra inválida. Usa solo: w, a, s, d o q")
        continue # Pide otra entrada si la tecla es inválida

    juego.paso(tecla)               # Procesa el movimiento del jugador
    juego.enemigo.paso(juego.comida) # La IA mueve su serpiente hacia la comida

    # Si la cabeza del jugador choca con cualquier parte de la serpiente enemiga
    if juego.serpiente[0] in juego.enemigo.serpiente:
        juego.viva = False # El jugador pierde

# Final del juego (fuera del bucle while)
juego.dibujar() # Dibuja el tablero una última vez para mostrar el estado final
print(f"\n{'═'*50}")
print(f"           GAME OVER - Puntaje Final: {juego.puntaje}")
print(f"{'═'*50}")

🎮 ¡Bienvenido! Tú eres la serpiente con cabeza **🟢** verde

╔════════════════════════════════════════════════════╗
║                       🐍 SNAKE                      ║
╠════════════════════════════════════════════════════╣
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ 🐍 ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ 🟢 ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬛ 🍎 ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬛ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ║
║ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜ ⬜